# Thesis Phase 1: Data Cleaning & Targeted Enrichment 🧹

**Objective:** Prepare the Raw Overpass API Data for the 12 Target Cities.
**Environment:** Local Repository Execution.

In [1]:
# 1. Setup & Imports
import pandas as pd
import glob
import os
import sys

# --- CONFIGURATION ---
BASE_DIR = r"E:\Github\repo-phaze7r\geospatial-tagging-thesis"
RAW_DIR = os.path.join(BASE_DIR, 'datareported', 'raw')
OUTPUT_DIR = os.path.join(BASE_DIR, 'datareported')

print(f"[*] Base Directory: {BASE_DIR}")
print(f"[*] Raw Data Directory: {RAW_DIR}")

# --- DIAGNOSTICS ---
print("\n--- DIAGNOSTICS ---")
if os.path.exists(RAW_DIR):
    print(f"[✓] Path exists: {RAW_DIR}")
    files = os.listdir(RAW_DIR)
    print(f"[✓] Found {len(files)} files in directory.")
    print(f"[i] Sample files: {files[:3]}")
else:
    print(f"[X] PATH NOT FOUND: {RAW_DIR}")
    print("\n[!] CRITICAL ERROR: The notebook cannot see your hard drive.")
    print("    Likely Cause: You are connected to 'Colab (Remote)' or a WSL/Docker container.")
    print(f"    Current Working Directory: {os.getcwd()}")
    print(f"    Operating System: {os.name} / {sys.platform}")
    if os.name == 'posix':
        print("    [!] You are on Linux (Remote/WSL). Windows paths (E:\\...) will NOT work.")
        print("    ACTION: Switch Kernel to 'Python 3.x.x' (Local Windows).")

[*] Base Directory: E:\Github\repo-phaze7r\geospatial-tagging-thesis
[*] Raw Data Directory: E:\Github\repo-phaze7r\geospatial-tagging-thesis\datareported\raw

--- DIAGNOSTICS ---
[✓] Path exists: E:\Github\repo-phaze7r\geospatial-tagging-thesis\datareported\raw
[✓] Found 25 files in directory.
[i] Sample files: ['attock_raw.csv', 'attock_raw.jsonl', 'chitral_raw.csv']


## 2. Load Raw Data

In [2]:
csv_files = glob.glob(os.path.join(RAW_DIR, "*_raw.csv"))
print(f"[*] Found {len(csv_files)} raw files.")

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    dfs.append(df)

if dfs:
    full_df = pd.concat(dfs, ignore_index=True)
    print(f"[*] Loaded Total {len(full_df)} rows.")
else:
    print("[!] No data found. Check diagnostics above.")

[*] Found 12 raw files.
[*] Loaded Total 54298 rows.


## 3. Data Preprocessing (Cleaning) 🧼

In [3]:
# A. Deduplication
initial_count = len(full_df)
if 'osm_id' in full_df.columns:
    full_df = full_df.drop_duplicates(subset=['osm_id'], keep='first')
else:
    full_df = full_df.drop_duplicates()
print(f"[-] Removed {initial_count - len(full_df)} duplicates.")

# B. Handling Empty Cells
full_df['description_raw'] = full_df['description_raw'].fillna("")
full_df['name'] = full_df['name'].fillna("Unknown POI")
print("[*] Nulls Handled.")

[-] Removed 4021 duplicates.
[*] Nulls Handled.


In [4]:
# Fallback Enrichment simulation
if 'enriched_description' not in full_df.columns:
    full_df['enriched_description'] = full_df['description_raw']
    print("[*] Created enriched_description column.")

[*] Created enriched_description column.


In [5]:
out_path = os.path.join(OUTPUT_DIR, 'final_enriched_dataset.csv')
full_df.to_csv(out_path, index=False)
print(f"[+] Saved: {out_path}")

[+] Saved: E:\Github\repo-phaze7r\geospatial-tagging-thesis\datareported\final_enriched_dataset.csv
